## MMTHE01 - Masters Thesis

### B. Thesis - Feature Engineering - Imputing missing fields
* Data Encoding
* Feature Scaling

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import pickle

from sklearn.preprocessing import LabelEncoder

In [2]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Importing the dataset

In [3]:
### Import the pickle saved dataset
with open("saved_data/final_dataset_encoded.pkl","rb") as f:
    dataset = pickle.load(f)

In [4]:
dataset.head()

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,revol_bal,policy_code,default_status,emp_title,title,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
0,500,4.0,33000.0,603.0,16.0,3461,1,0,80998.0,5404.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,6000,19.0,12000.0,122.0,34.0,5251,1,0,54628.0,1527.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,14700,23.0,59000.0,257.0,27.0,14301,1,0,34339.0,7118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,5000,20.0,98000.0,93.0,34.0,9568,1,0,51372.0,5870.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,3750,27.0,7280.0,550.0,14.0,0,1,0,48332.0,5780.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [5]:
dataset.shape

(226067, 116)

In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226067 entries, 0 to 226066
Columns: 116 entries, loan_amnt to initial_list_status_w
dtypes: category(1), float64(109), int64(4), object(2)
memory usage: 198.6+ MB


### 4.1 Split the data into Train-Test

#### 4.1.1 Separate the features and the label

In [7]:
X = dataset.drop(columns=['default_status'])
y = dataset['default_status']

#### 4.1.2 Feature Scaling

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
sc = StandardScaler()
X_scaled = pd.DataFrame(sc.fit_transform(X), columns=X.columns)

#### 4.1.3 Keep OHE features

In [10]:
cols_for_ohe = ['emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_month', 'earliest_cr_line_month', 'initial_list_status']

In [11]:
ohe_cols = {}
for col in cols_for_ohe:
    col_values = X_scaled.filter(regex='^'+col, axis=1).columns.tolist()
    ohe_cols[col] = col_values

In [12]:
#ohe_cols

### Step 4.2 Feature Selection

#### 4.2.1 Remove features with low variance

In [13]:
from sklearn.feature_selection import VarianceThreshold

In [14]:
vt = VarianceThreshold(threshold=0.0)
X_vt = vt.fit_transform(X_scaled)
selected_features_vt = X.columns[vt.get_support()]
print(f"After Variance Threshold: {len(selected_features_vt)} features retained")

After Variance Threshold: 114 features retained


In [15]:
# Features that were removed
removed_features_vt = X.columns[~vt.get_support()]
removed_features_vt

Index(['policy_code'], dtype='object')

In [16]:
selected_features_vt_hold = []

for key, value in ohe_cols.items():
    for removed_feature in removed_features_vt:
        if removed_feature in value:
            for column in value:
                selected_features_vt_hold = [x for x in selected_features_vt.tolist() if x != column]
                
if selected_features_vt_hold == []:
    selected_features_vt_hold = selected_features_vt.tolist()

In [17]:
unique_list_vt=[]
for key, values in ohe_cols.items():
    for selected_feature in selected_features_vt:
        if selected_feature in values:
            for value in values:
                if value not in selected_features_vt:
                    if value not in unique_list_vt:
                        unique_list_vt.append(value)

In [18]:
selected_features_vt = selected_features_vt_hold + unique_list_vt

In [19]:
len(selected_features_vt)

114

#### 4.2.2 Using ANOVA F-test for feature selection

In [20]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

In [21]:
X_scaled = X_scaled[selected_features_vt]

In [22]:
X_scaled.head(2)

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,revol_bal,emp_title,title,dti,delinq_2yrs,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
0,-1.582429,-0.996229,-0.559961,0.503531,-0.472873,-0.560385,1.641204,2.121159,-1.306363,-0.353307,...,-0.279751,-0.297284,-0.291,-0.299035,-0.289692,-0.308314,-0.3241,-0.329126,1.455052,-1.455052
1,-0.983936,1.386170,-0.821678,-1.233354,0.722534,-0.484443,0.495935,-0.983326,-1.129800,-0.353307,...,-0.279751,-0.297284,-0.291,-0.299035,-0.289692,-0.308314,-0.3241,-0.329126,1.455052,-1.455052


In [23]:
f_values, p_values = f_classif(X_scaled, y)
anova_scores = pd.DataFrame({
    'Feature': X_scaled.columns,
    'F-Score': f_values,
    'p-Value': p_values
}).sort_values(by='F-Score', ascending=False)

anova_scores.tail()

,Feature,F-Score,p-Value
56,emp_length_2 years,0.039464,0.842532
110,earliest_cr_line_month_Oct,0.036448,0.848594
108,earliest_cr_line_month_May,0.028743,0.865374
57,emp_length_3 years,0.002024,0.964114
3,zip_code,0.000085,0.992641


In [24]:
# Add a column indicating selection status
# -----------------------------
# Feature selection thresholds
# -----------------------------
min_f_score = 1    # keep features with F-score >= 1
max_p_value = 0.05  # keep features with p-value <= 0.05

anova_scores['Selected'] = (
    (anova_scores['F-Score'] >= min_f_score) &
    (anova_scores['p-Value'] <= max_p_value)
)

In [25]:
# -----------------------------
# Separate selected and removed for clarity (optional)
# -----------------------------
selected_df = anova_scores[anova_scores['Selected']].copy()
selected_df.head(1)

,Feature,F-Score,p-Value,Selected
1,sub_grade,11595.736552,0.0,True


In [26]:
removed_df = anova_scores[~anova_scores['Selected']].copy()
removed_df.head(1)

,Feature,F-Score,p-Value,Selected
4,addr_state,3.584217,0.058332,False


In [27]:
removed_features_annova = removed_df['Feature'].tolist()

In [28]:
selected_features_annova = selected_df['Feature'].tolist()
len(selected_features_annova)

81

In [29]:
selected_features_annova_hold = []

for key, value in ohe_cols.items():
    for removed_feature in removed_features_annova:
        if removed_feature in value:
            for column in value:
                selected_features_annova_hold = [x for x in selected_features_annova if x != column]

if selected_features_annova_hold == []:
    selected_features_annova_hold = selected_features_annova.tolist()

In [30]:
len(selected_features_annova_hold)

81

In [31]:
#selected_features_annova

In [32]:
unique_list=[]
for key, values in ohe_cols.items():
    for selected_feature in selected_features_annova:
        if selected_feature in values:
            for value in values:
                if value not in selected_features_annova:
                    if value not in unique_list:
                        unique_list.append(value)

In [33]:
selected_features_final = selected_features_annova_hold + unique_list

In [34]:
len(selected_features_final)

109

#### 4.2.3 Export encoded data

In [35]:
selected_features_final.sort()

In [36]:
selected_features_final.append('default_status')

In [37]:
final_dataset = dataset[selected_features_final]

In [38]:
ohe_cols_final = []
for col in cols_for_ohe:
    col_values = final_dataset.filter(regex='^'+col, axis=1).columns.tolist()
    for values in col_values:
        ohe_cols_final.append(values)
#ohe_cols_final

In [39]:
ohe_check = []
ohe_cols_final = final_dataset.filter(regex='^'+col, axis=1).columns.tolist()
for ohe_feature in ohe_cols_final:
    if ohe_feature not in selected_features_final:
        print(ohe_feature + ' is was not selected')
        ohe_check.append(ohe_feature)
if ohe_check==[]:
    print('All one-hot-encoded features were retained')

All one-hot-encoded features were retained


In [40]:
# Save datasets in one file
with open("saved_data/final_dataset.pkl", "wb") as f:
    pickle.dump((final_dataset), f)

In [41]:
final_dataset.shape

(226067, 110)

In [42]:
final_dataset.head(5).to_csv('feature_features_data_sample.csv',index=False)

In [43]:
# Export final features selected
with open("saved_data/selected_features_final.pkl", "wb") as f:
    pickle.dump((selected_features_final), f)